<a href="https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/assignments/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The page is a prime candidate for a Refresh if an impression, visibility is high (>=500) as well as the avg position (<=10) but the CTR is low. It shows that it might be helpful to change for example the title so that the users might click on this page more often.

- Reason code: high_volume_low_ctr
- Action: Refresh Snippet/Content

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import pandas as pd
import numpy as np

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

TARGET_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

In [ ]:
#splitting the data into past (features) and future (target) like before
# avg_position = 0 means "no data", not position 0 that's why we use NULLIF

query = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_past,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_past,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN NULLIF(gsc_avg_position, 0) END) AS avg_pos_past,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_future
    FROM read_parquet('{TARGET_MONTH}')
    GROUP BY 1, 2
    HAVING imp_past >= 100
"""
df = con.sql(query).df()

# Calculate CTR (Clicks / Impressions * 100, matching the x100 percentage format)
df['ctr_past'] = np.where(df['imp_past'] > 0, (df['clicks_past'] / df['imp_past']) * 100, 0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
print("Signal 1: Volume Buckets (n)")
df['vol_bucket'] = pd.cut(df['imp_past'], bins=[0, 500, 5000, np.inf], labels=['Low (<500)', 'Med (500-5k)', 'High (>5k)'])
print(df.groupby('vol_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr_past', 'mean')
))

print("\n Signal 2:  Position vs CTR Buckets (n)")
df['pos_bucket'] = pd.cut(df['avg_pos_past'], bins=[0, 3, 10, 50, np.inf], labels=['Top 3', 'Page 1 (4-10)', 'Page 2-5', 'Deep'])
print(df.groupby('pos_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    avg_ctr=('ctr_past', 'mean')
))

Signal 1: Volume Buckets (n)
                  n   avg_ctr
vol_bucket                   
Low (<500)    35774  0.265403
Med (500-5k)  36343  0.318177
High (>5k)     5423  0.293614

 Signal 2:  Position vs CTR Buckets (n)
                   n   avg_ctr
pos_bucket                    
Top 3           9197  0.411871
Page 1 (4-10)  37447  0.335484
Page 2-5       29294  0.212892
Deep            1602  0.039326


Signal Checks:

Signal 1 (Volume behind quick-win): We bucket past impressions. Verdict: MIXED. The data shows that having the highest volume (>5k) does not guarantee the highest CTR. This proves our rule is necessary: high impressions don't automatically mean clicks. Filtering for those high-volume, low-CTR pages targets a real inefficiency.

Signal 2 (CTR-vs-position): We bucket CTR by position. Verdict: CONFIRMED. There is a perfect degradation of CTR as average position drops. Breaking this natural trend (e.g., ranking on Page 1 but having a near-zero CTR) flags a true issue.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#target label - did impressions drop by > 20%?
df['is_declining_label'] = (df['imp_future'] < 0.8 * df['imp_past']).astype(int)
base_rate = df['is_declining_label'].mean()

#score heavily prioritizes pages with High Impressions, Good Position, but Bad CTR.
is_page_1 = (df['avg_pos_past'] <= 10).astype(int)
is_low_ctr = (df['ctr_past'] < 2.0).astype(int)

#transparent score without fitted weights
df['score'] = df['imp_past'] * is_page_1 * is_low_ctr

df['reason_code'] = np.where(df['score'] > 0, 'high_vol_low_ctr', 'none')
df['action'] = np.where(df['score'] > 0, 'Refresh Snippet', 'Ignore')

queue = df.sort_values('score', ascending=False).copy()

k = 50
precision_at_k = queue['is_declining_label'].head(k).mean()
print(f"Base Rate (Randomly picking pages): {base_rate:.3f}")
print(f"Baseline Precision@{k}: {precision_at_k:.3f}")

#write to CSV
os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(csv_path, index=False)
print(f"Saved ranked queue to {csv_path}")

Base Rate (Randomly picking pages): 0.285
Baseline Precision@50: 0.220
Saved ranked queue to work/outputs/baseline_action_score.csv


The evaluation showed that our baseline rule's precision is lower than the base rate (randomly picking pages). This likely occurs because our scoring logic pushes pages with the absolute highest impressions to the top of the queue. In a short 15-day window, it is rare for these highly stable, top-performing pages to experience a traffic drop of at least 20%.

In [ ]:
df['is_declining_label'] = (df['imp_future'] < 0.8 * df['imp_past']).astype(int)
base_rate = df['is_declining_label'].mean()

# we are now looking for pages that are "on the edge" of the first page (position 7-10) and may decrease
is_edge_of_page_1 = ((df['avg_pos_past'] >= 7) & (df['avg_pos_past'] <= 10)).astype(int)

#and lower CTR
is_low_ctr = (df['ctr_past'] < 1.0).astype(int)

# we are looking for pages with high volume
has_decent_volume = (df['imp_past'] >= 500).astype(int)

is_at_risk = has_decent_volume * is_edge_of_page_1 * is_low_ctr

# New Score: we promote pages with the wors CTR
df['score'] = np.where(is_at_risk > 0, 1.0 / (df['ctr_past'] + 0.01), 0)

df['reason_code'] = np.where(df['score'] > 0, 'edge_page1_low_ctr', 'none')
df['action'] = np.where(df['score'] > 0, 'Refresh Snippet', 'Ignore')

queue = df.sort_values('score', ascending=False).copy()

k = 50
precision_at_k = queue['is_declining_label'].head(k).mean()
print(f"Base Rate (Randomly picking pages): {base_rate:.3f}")
print(f"Baseline Precision@{k}: {precision_at_k:.3f}")

os.makedirs('work/outputs', exist_ok=True)
csv_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(csv_path, index=False)
print(f"Saved ranked queue to {csv_path}")

Base Rate (Randomly picking pages): 0.285
Baseline Precision@50: 0.400
Saved ranked queue to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
columns_to_show = ['content_hash_id', 'score', 'imp_past', 'ctr_past', 'avg_pos_past', 'imp_future', 'action', 'reason_code', 'is_declining_label']
display(queue[columns_to_show].head(10))

,content_hash_id,score,imp_past,ctr_past,avg_pos_past,imp_future,action,reason_code,is_declining_label
52285,content_9efb161830e29271,100.0,644.0,0.0,9.258326,2295.0,Refresh Snippet,edge_page1_low_ctr,0
34971,content_4fff303667fa35fb,100.0,644.0,0.0,7.256783,474.0,Refresh Snippet,edge_page1_low_ctr,1
30375,content_2f951e758d44a3cd,100.0,510.0,0.0,9.503820,163.0,Refresh Snippet,edge_page1_low_ctr,1
26805,content_173b3082e7debaea,100.0,1550.0,0.0,7.347923,1697.0,Refresh Snippet,edge_page1_low_ctr,0
23511,content_073ce5ca88a1d8a7,100.0,826.0,0.0,8.367272,981.0,Refresh Snippet,edge_page1_low_ctr,0
43501,content_ed0f7ee885fe066b,100.0,561.0,0.0,7.139147,591.0,Refresh Snippet,edge_page1_low_ctr,0
1637,content_997ab46faed74304,100.0,1465.0,0.0,8.268090,1513.0,Refresh Snippet,edge_page1_low_ctr,0
22376,content_30111561b719b28e,100.0,980.0,0.0,9.691736,462.0,Refresh Snippet,edge_page1_low_ctr,1
25252,content_d97b74fab9a87e3d,100.0,607.0,0.0,9.260795,325.0,Refresh Snippet,edge_page1_low_ctr,1
62255,content_9b535f14a0d47dfe,100.0,2726.0,0.0,8.573652,3828.0,Refresh Snippet,edge_page1_low_ctr,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The refined heuristic beats the base rate (Precision@50 of 0.400), a review of the Top 10 ranked pages reveals obvious false positives. Specifically, there are pages flagged for a "Refresh Snippet" intervention where imp_future actually turned out to be higher than imp_past.
Our baseline relies on rigid, static thresholds. It is "blind" to other context and fails to account for the complexity. The true interaction between volume, exact position, and clickability is too complex for a hardcoded mathematical formula. The problem may also be the fact that it takes under consideration only one month, which may be too short to really see if the impressions decreases or not.

To minimize false positives, we need to evaluate multiple features simultaneously. Tree-based algorithms, such as XGBoost or LightGBM, will be ideal for the next stage to capture non-linear patterns.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.